## Webpage Extraction and Embedding (PolyU SAO)

### 1. Extracting raw text data

In [1]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

In [2]:
sao_URL = "https://www.polyu.edu.hk/sao/"
start_idx, stop_idx = 36450, -900

docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "lxml")                  # Strip the html syntax
    text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess newlines
    return text

sao_loader = RecursiveUrlLoader(
    url=sao_URL,
    base_url=sao_URL,
    prevent_outside=True,
    exclude_dirs=[
        sao_URL+"News-and-Events", 
        sao_URL+"About-SAO", 
        sao_URL+"Sitemap", 
        sao_URL+"Search-Result",
        sao_URL+"National-Education",
        sao_URL+"Personal-Information-Collection-Statement",
        sao_URL+"Student-Development-Section",
        sao_URL+"student-development-section",
        sao_URL+"Counselling-and-Wellness-Section/PolyU-Asian-Universities-Water-Polo-Invitational-Tournament",
        sao_URL+"Counselling-and-Wellness-Section/Wellness-Centre",
        sao_URL+"Counselling-and-Wellness-Section/Sports-Development",
        sao_URL+"Counselling-and-Wellness-Section/Programmes-and-Activities",
        sao_URL+"Student-Resources-and-Support-Section/Outstanding-Student-Academy",
        sao_URL+"Careers-and-Placement-Section/Gallery-and-Publications",
        sao_URL+"Non-local-Student-Services/Settling-in",
        sao_URL+"Non-local-Student-Services/Event-Highlight",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = sao_loader.lazy_load()
for doc in docs_lazy:
    doc.page_content = doc.page_content[start_idx:stop_idx]
    for key in unwanted_metadata:
        del doc.metadata[key]
    docs.append(doc)

In [3]:
print(f"Extracted number of webpages in SAO: {len(docs)}")
print(doc)

'''
idx = 5
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in SAO: 157
page_content='                

Quick Access

Start main content

													Home
												

													Student Resources and Support
												

													Financial Assistance
												

													PolyU Financial Assistance Schemes
												

													PolyU Emergency Financial Assistance Scheme
												

PolyU Emergency Financial Assistance Scheme

Eligibility

All local full-time degree and sub-degree students (For needy postgraduates, they may be considered for The Croucher Foundation Fund for Students with Emergency Needs.) facing dire financial difficulties caused by recent unforeseen circumstances such as sudden unemployment/serious illness/accident/death of the sole breadwinner of the family, natural or man-made disaster, etc. at any time of the academic year, may apply for emergency financial assistance.

Source of Funds

Available funds donated by philanthropists are:
√ Hong Kong Rotary Club Students’ Loan F

"\nidx = 5\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1)
    chunk.metadata["chunk_id"] = f"PolyU_SAO_{url_end}_chunk_{i}"

In [5]:
print(chunks[10])

page_content='Quick Access

Start main content

													Home
												

													Student Resources and Support
												

													Supporting Students with Special Educational Needs
												

													Telling us about your Special Needs
												

Telling us about your Special Needs

Telling us about your Special Needs                                    

 

Declare your Special Needs

Read More

 

 

Useful Information

Read More

 

 

Our Services

Our Services                                        

                                                            Welcome Pack
                                                        

                                                            Types of Special Needs
                                                        

                                                            Campus Resources' metadata={'source': 'https://www.polyu.edu.hk/sao/Student-Resources-and-Support-Section/Special-Needs-Support

### 3. Document Embedding in Chroma

In [6]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_sao_webpage" if not SINGLE else "vaa_documents"

In [7]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="./chroma_db")
collection = client.get_collection(name=collection_name)

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_26940/3659695745.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!


In [8]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i)]
    )

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Added 650 chunks into ChromaDB to vaa_documents


/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_26940/2423177955.py:14: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


### 4. Simple Testing

In [9]:
query = "How can I apply for residential hall in PolyU?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: Quick Access

Start main content

													Home
												

													Student Resources and Support
												

													Residential Life
												

													Hall Admission
												

													Admission Policies for Undergraduates
												

Admission Policies for Undergraduates

Admission Policy

General Information on Application

Special Readmission Scheme (SRS)

Readmission Scheme of CURI Residential College (RSCRC)

 
Admission Policy
A. Eligibility
The PolyU Student Halls were established with major funding support from the University Grants Committee (UGC), hence, eligibility to hall residence has to be set with reference to the UGC guidelines. The University has come up with a set of policies to govern admission of students to hall residence. The following groups of students are eligible for hall residence:...
Source: https://www.polyu.edu.hk/sao/Student-Resources-and-Support-Section/Residential-Life/Hall-Admission/Admission-Policies-Und